# control48 — GEVIS만 증류하고 원본 Head 유지

이 노트북은 사용자 실행용입니다. 에이전트는 제조 데이터·가중치를 실행하지 않았습니다.
고정된 방식: 내부48차원·Transformer2층·Cross-Attention Student → **변경 없는 원본 판별 Head**.
Teacher 질문 벡터와 Head는 학습하지 않습니다. 5유형 argmax를 최종 답변에서만 다음 3유형으로 매핑합니다.

- 수준+추세 → 수준·추세
- 변동성 → 변동성
- 국소 변화+지연 → 국소 변화·지연

확률 합산·새 3출력 Head 학습·Head 축소는 없습니다.
BOOLEAN/ORDINAL gate와 원래 저장 정규화·TRAIN 참조를 유지합니다.
Validation으로 epoch·선택을 고정한 뒤 Test를 한 번만 평가합니다.
모든 결과와 특징 캐시는 비공개로 취급합니다.


In [ ]:
from pathlib import Path
import os, sys
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
WORKSPACE_PATH = None
SOURCE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'src/heads/distilation').is_dir()), None)
if SOURCE_ROOT is None:
    raise FileNotFoundError('GEVIS 프로젝트 또는 notebooks/distilation에서 실행하세요.')
ROOT = SOURCE_ROOT if WORKSPACE_PATH is None else Path(WORKSPACE_PATH).expanduser().resolve()
SOURCE = SOURCE_ROOT / 'src'
for name, module in tuple(sys.modules.items()):
    if name in ('heads', 'pecvd') or name.startswith(('heads.', 'pecvd.')):
        for location in [getattr(module, '__file__', None), *getattr(module, '__path__', ())]:
            if location and not Path(location).resolve().is_relative_to(SOURCE):
                raise RuntimeError('다른 프로젝트 모듈이 로딩되어 있습니다. 해당 커널의 결과를 보존한 뒤 새 커널을 사용하세요.')
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from heads import distilation as kd
import json, importlib
import heads.control48_fast as fast_module
fast = importlib.reload(fast_module)


## 1. 검증한 legacy 설정과 원본 저장본

기본값은 사용자가 확인한 control48 설정입니다.
Teacher는 기존 평가의 원본 5유형 Softmax 패키지입니다. 다른 유형으로 대체하지 않습니다.
TEACHER_FORMAT='domain'은 현재 ALD/CVD_evaluate 패키지, 'detailed'는 기존 detailed STEP3 패키지입니다.
후자의 원래 입력·타깃 규약도 유지합니다. 기본은 현재 평가 패키지입니다.

Student가 아직 커널에 남아 있으면 LIVE_CONTROL48·LIVE_TRAINING_RECORD·LIVE_ORIGINAL_HEAD로 명시해 추가 학습 없이 복사할 수 있습니다.
예: 기존 실험의 candidates['control48'][0], records['control48'], original_head.
검증 자료가 없으면 아래 실행에서 **Student만** 학습하고 즉시 병합본을 저장합니다.


In [ ]:
DOMAIN = 'PEALD'  # 'PECVD'도 지원
TEACHER_FORMAT = 'domain'
TEACHER_CHECKPOINT = None  # 기존 original-five answer_head.pt 경로; None은 유일 Softmax 또는 번호 선택
ALLOW_LEGACY_CVD = DOMAIN == 'PECVD'
ALLOW_LEGACY_ALD = DOMAIN == 'PEALD'
CONFIG = kd.Config(
    seed=42, encoder_width=48, encoder_layers=2,
    learning_rate=.001, temperature=2., alpha=.5, feature_weight=1.,
    epochs=40, patience=8, batch_cases=8, max_seconds=1200.,
    center_batch_size=8,
    train_cases_per_step=64, validation_cases_per_step=64, test_cases_per_step=64,
    benchmark_windows=16, repeats=5, warmup=2,
)
# 데이터 예산은 Teacher 저장 규약 안에서 평가할 사례 수를 제한합니다.
# 전체 Run/실제 고장 평가가 아닌 bounded controlled generation 평가입니다.
PREPARED_IN_MEMORY = None  # 같은 Teacher의 원래 PREPARED가 있으면 지정. 정규화·분할 일치 검사 후 재사용
LIVE_CONTROL48 = None
LIVE_TRAINING_RECORD = None
LIVE_ORIGINAL_HEAD = None

RUN_OPTUNA = False  # True이면 Validation 전용 선택 탐색. 구조·Head·답변 매핑은 고정
N_TRIALS = 12       # 기본 trial당 최대 학습 예산 1200초 + 평가/추론 시간. 전체 실행은 오래 걸릴 수 있음
SEARCH_TIMEOUT_SECONDS = None
RUN_FINAL_TEST = True  # Validation 선택 동결 이후에만 Test 생성/평가

# 추론 배치는 학습 batch_cases/center_batch_size와 별도로 Validation에서 선택합니다.
INFERENCE_BATCHES = (8, 16, 32)
MEAN_TARGET_MS = 50.
CHECK_ALL_SCORED_CASES = True


In [ ]:
TEACHER = kd.load_teacher(ROOT, domain=DOMAIN, checkpoint=TEACHER_CHECKPOINT,
    format=TEACHER_FORMAT, allow_legacy_cvd=ALLOW_LEGACY_CVD, allow_legacy_ald=ALLOW_LEGACY_ALD)
display(kd.mapping_table())
print('원본 Head 파라미터 수:', sum(p.numel() for p in TEACHER.module.FUSION_ANSWER_HEAD.parameters()))
print('Teacher native 경로:', len(TEACHER.encoders))


## 2. 원래 입력·정규화·참조에 결합된 데이터 준비

새 KD 학습에 필요한 원본 Window만 기존 전처리 경로로 준비합니다.
저장된 Teacher의 센서 순서·TRAIN mean/std·native 길이/간격·Run 분할·source identity와 다르면 중단합니다.
정상 참조는 Teacher 패키지의 MIN/MAX를 그대로 사용합니다. 재구성·재학습하지 않습니다.
이미 PREPARED가 있으면 위 PREPARED_IN_MEMORY로 전달해 준비를 반복하지 않을 수 있습니다.

Student 입력은 제공한 그대로:
values / lower / upper / (values-lower)/scale / (values-upper)/scale,
scale=max(upper-lower,1), 부호 유지, 전체 asinh 후 [sensors,5,time].


In [ ]:
# Initialize/align Teacher text and numeric modules before any data preparation.
# Reload only the updated Study class; preserve loaded weights and prepared windows.
import importlib
study_module = importlib.import_module('heads.distilation.study')
kd.Study = importlib.reload(study_module).Study
EXPERIMENT = kd.Study(ROOT, TEACHER, CONFIG)
existing_prepared = PREPARED_IN_MEMORY if PREPARED_IN_MEMORY is not None else globals().get('PREPARED')
if existing_prepared is None:
    PREPARED = kd.prepare_for_teacher(ROOT, TEACHER)
else:
    PREPARED = kd.attach_prepared(TEACHER, existing_prepared)


## 3. Student만 학습 또는 검증한 live Student 병합

legacy 손실:
0.5×정답 CE + 0.5×T² KL + 1.0×특징 Smooth L1.
특징은 원본 Head의 feature_scale로 정규화하고 마지막 경계·시간 통계10개는 특징 손실에서 제외합니다.
Teacher 특징·확률은 고정하고 원본 Head를 통과하는 gradient는 Student로만 전파합니다.
학습용 Teacher row는 새 .cache/distilation 실행 폴더에 저장해 메모리 누적을 제한합니다.

Epoch selection keeps the original-five legacy Validation criterion. The fixed three-type mapping is applied only after epoch selection.


In [ ]:
if LIVE_CONTROL48 is None:
    HISTORY = EXPERIMENT.fit()
    display(HISTORY)
else:
    if LIVE_TRAINING_RECORD is None or LIVE_ORIGINAL_HEAD is None:
        raise ValueError('live Student의 legacy 학습 기록과 원본 Head도 지정하세요.')
    EXPERIMENT.adopt_student(LIVE_CONTROL48, LIVE_TRAINING_RECORD, LIVE_ORIGINAL_HEAD)
    print('검증한 live control48 복사 완료. 추가 fitting 없음.')


## 4. Validation 세 과제 F1과 실제 전체 요청 지연

Teacher와 control48 모두 동일한 최종 3유형 매핑으로 비교합니다.
세부 5유형 CHOICE F1도 함께 표시해 매핑에 따른 과제 변경을 구분합니다.
unknown 유형은 유형 평가에서 제외하고 gate 미탐·정상 오탐은 기존 지표에 유지합니다.
클래스별 support·제외·생성 불가도 확인하세요. F1만 높고 지원이 빠진 결과는 완전한 검증이 아닙니다.

시간은 준비/로딩/워밍업을 제외한 실제 단일 Window 전체 요청의 평균·p95, 단위 ms입니다.
원래 raw 좌표 복원 → 정규화·참조 조회·GEVIS·원본 Head·조건부 답변·최종 매핑과 GPU 동기화를 포함합니다.
전체 센서 시간이므로 센서 수로 나눠 단일 센서 지연이라고 부르지 않습니다.

이 절의 시간은 Teacher/control48 순차 기준값입니다. 아래 병합 저장 절에서 배치 추론을 검증하고 최종 실행 배치를 선택합니다.
이 노트북 하나를 처음부터 실행하면 됩니다. 다른 노트북의 커널 객체는 필요 없습니다.


In [ ]:
VALIDATION = EXPERIMENT.report('validation')
display(VALIDATION['metrics'])
display(pd.DataFrame(VALIDATION['support']).T)
display(pd.DataFrame(VALIDATION['coverage']['support']))
display(pd.DataFrame(VALIDATION['coverage']['generation']))
PARITY = EXPERIMENT.parity_report()
display(PARITY)
VALIDATION_TIMING = EXPERIMENT.benchmark('validation')
display(VALIDATION_TIMING['summary'])


## 5. 선택 실행 — Optuna

RUN_OPTUNA=False이면 검증한 설정을 그대로 유지합니다.
True이면 학습률·temperature·alpha·feature_weight만 탐색합니다. 48차원·2층·원본 Head·최종 그룹·표본은 고정합니다.
목표는 가장 낮은 Validation 과제 F1 최대화와 실제 요청 p95 최소화의 두 가지입니다.
세 과제 목표와 원본 클래스 지원을 만족한 trial 중 p95가 작은 것을 고릅니다.
만족 trial이 없으면 가장 약한 과제 F1이 높은 것을 고르되 **목표 미달**로 명시합니다.
Test 값은 탐색·선택에 쓰지 않습니다. Optuna 탐색은 검증한 고정 설정과 구분되는 새 실험입니다.
패키지가 없으면 현재 커널 환경에 pip install -e ".[fdc,notebook,distilation]"로 준비하세요.


In [ ]:
if RUN_OPTUNA:
    from heads.distilation.search import optimize
    SEARCH = optimize(EXPERIMENT, n_trials=N_TRIALS, timeout=SEARCH_TIMEOUT_SECONDS)
    display(SEARCH['trials'])
    print('선택 trial:', SEARCH['selected_trial'], '| Validation 목표 충족:', SEARCH['validation_qualified'])
    VALIDATION = SEARCH['validation']
    VALIDATION_TIMING = SEARCH['timing']
    display(VALIDATION['metrics'])
    display(VALIDATION_TIMING['summary'])


## 6. Validation 선택 고정·병합 저장·Teacher 없는 재로딩

선택 모델을 새로운 artifacts/gevis_distilation/.../control48.pt에 명시적으로 저장합니다.
원본 Teacher/Head 파일·기존 완료 포인터는 덮어쓰지 않습니다.
병합본은 원본 Head의 state를 그대로 포함하며 Student+Head만 로딩합니다. Teacher/E5 로딩이 필요 없습니다.
저장 후 동일 raw 입력의 결정 일치도 확인합니다.

저장·재로딩 직후 같은 Validation raw Window로 순차/배치8·16·32를 비교합니다.
원본5유형·BOOLEAN·ORDINAL 결정과 확률이 일치해야만 시간을 측정합니다.
평균 목표를 만족하는 후보 중 p95가 작은 배치를 선택하고, 만족 후보가 없으면 목표 미달로 표시합니다.
Test는 이 실행 배치를 고르는 데 사용하지 않습니다.


In [ ]:
SELECTION = EXPERIMENT.freeze_selection()
display(pd.DataFrame([SELECTION]))
MERGED_CHECKPOINT = EXPERIMENT.save()
print('병합 저장본:', MERGED_CHECKPOINT)
MERGED = kd.load_merged(MERGED_CHECKPOINT, device=TEACHER.module.DEVICE)
display(EXPERIMENT.merged_parity_report(MERGED))

# 모델 선택은 이미 동결했습니다. 실행 배치만 Validation에서 선택합니다.
VALIDATION_TIMING = EXPERIMENT.benchmark('validation', merged=MERGED)
print('Teacher/control48 순차 기준 시간:')
display(VALIDATION_TIMING['summary'])
VALIDATION_REQUESTS = fast.heldout_requests(EXPERIMENT, 'validation', windows=CONFIG.benchmark_windows)
FAST_VALIDATION = fast.benchmark_requests(MERGED, VALIDATION_REQUESTS,
    batches=INFERENCE_BATCHES, repeats=CONFIG.repeats, warmup=CONFIG.warmup, split='validation')
display(FAST_VALIDATION['parity'])
display(FAST_VALIDATION['summary'])
EXECUTION_SELECTION = fast.choose_batch(FAST_VALIDATION, target_ms=MEAN_TARGET_MS)
display(pd.DataFrame([EXECUTION_SELECTION]))
SESSION = fast.FastSession(MERGED, center_batch_size=EXECUTION_SELECTION['center_batch_size'])
if CHECK_ALL_SCORED_CASES:
    FAST_VALIDATION_F1 = fast.verify_scored_cases(EXPERIMENT, MERGED, split='validation',
        center_batch_size=EXECUTION_SELECTION['center_batch_size'])
    display(FAST_VALIDATION_F1['metrics'])

# 새 병합본 옆에만 기록합니다. 기존 모델 파일은 변경하지 않습니다.
REPORT_DIR = MERGED_CHECKPOINT.parent
VALIDATION['metrics'].to_csv(REPORT_DIR / 'validation_f1.csv', index=False)
VALIDATION_TIMING['summary'].to_csv(REPORT_DIR / 'validation_serial_latency_ms.csv', index=False)
FAST_VALIDATION['summary'].to_csv(REPORT_DIR / 'validation_latency_ms.csv', index=False)
FAST_VALIDATION['parity'].to_csv(REPORT_DIR / 'validation_runtime_parity.csv', index=False)
if CHECK_ALL_SCORED_CASES:
    FAST_VALIDATION_F1['metrics'].to_csv(REPORT_DIR / 'validation_runtime_f1.csv', index=False)
(REPORT_DIR / 'execution.json').write_text(json.dumps(EXECUTION_SELECTION, ensure_ascii=False, indent=2), encoding='utf-8')
(REPORT_DIR / 'control48_fast.py').write_bytes((SOURCE / 'heads/control48_fast.py').read_bytes())


## 7. 최종 Test — 선택 이후 한 번만

선택된 Student의 최종 확인입니다. 여기서 나온 점수로 architecture·loss·그룹·epoch를 다시 고르지 않습니다.
이미 과거에 열람한 Test라면 독립 최종 평가라고 부르지 마세요. 사용자 과거 열람 여부는 이 코드가 복원할 수 없습니다.

Test 시간은 Validation에서 선택한 고정 배치 하나로 측정합니다.
세 과제 F1은 원래 보고서와 실제 순차/배치 런타임 일치 확인 표를 구분합니다.
F1 일치 확인은 원래 정규화 입력·scored centers, 지연은 전체 센서 raw Window를 사용합니다.


In [ ]:
if RUN_FINAL_TEST:
    TEST = EXPERIMENT.evaluate_test()
    display(TEST['metrics'])
    display(pd.DataFrame(TEST['support']).T)
    TEST_TIMING = EXPERIMENT.benchmark('test', merged=MERGED)
    print('Teacher/control48 순차 기준 시간:')
    display(TEST_TIMING['summary'])
    TEST_REQUESTS = fast.heldout_requests(EXPERIMENT, 'test', windows=CONFIG.benchmark_windows)
    FAST_TEST = fast.benchmark_requests(MERGED, TEST_REQUESTS,
        batches=(EXECUTION_SELECTION['center_batch_size'],), repeats=CONFIG.repeats,
        warmup=CONFIG.warmup, split='test')
    print('고정 배치 최종 Test 시간:')
    display(FAST_TEST['summary'])
    display(FAST_TEST['parity'])
    if CHECK_ALL_SCORED_CASES:
        FAST_TEST_F1 = fast.verify_scored_cases(EXPERIMENT, MERGED, split='test',
            center_batch_size=EXECUTION_SELECTION['center_batch_size'])
        display(FAST_TEST_F1['metrics'])
        FAST_TEST_F1['metrics'].to_csv(REPORT_DIR / 'test_runtime_f1.csv', index=False)
    TEST['metrics'].to_csv(REPORT_DIR / 'test_f1.csv', index=False)
    TEST_TIMING['summary'].to_csv(REPORT_DIR / 'test_serial_latency_ms.csv', index=False)
    FAST_TEST['summary'].to_csv(REPORT_DIR / 'test_latency_ms.csv', index=False)
    FAST_TEST['parity'].to_csv(REPORT_DIR / 'test_runtime_parity.csv', index=False)
    print('Validation/Test F1·배치 추론 평균/p95 보고서 로컬 저장 완료.')


## 8. 병합본으로 추론

이 셀은 이미 로딩한 병합본의 실행 예시입니다. 판정 입력에 이상을 자동 주입하지 않습니다.
실제 입력은 저장된 sensor_columns 순서·native 길이·recipe/loop/elapsed 문맥과 일치해야 합니다.


In [ ]:
REQUEST = EXPERIMENT.request(EXPERIMENT.dataset('validation')['cases'][0], raw=True)
REPLY = SESSION.process_window(**REQUEST)
display(pd.DataFrame(REPLY['answers']))
